# Dengue, saneamento e clima — São Paulo, 2010 × 2022

Comparação dos dois anos censitários usando tabelas Silver do Databricks. O notebook mantém notificações como medida de demanda de vigilância e acrescenta casos confirmados, casos graves e óbitos por dengue.

> **Escopo:** residentes em SP, população e saneamento municipais do SIDRA e clima agregado das estações de SP. As categorias censitárias de saneamento são preservadas como publicadas em cada ano e não medem evolução harmonizada.

## 1. Configuração

Execute antes `02_silver_sinan_sp`, `02_silver_dengue_sidra_sp`, `02_silver_inmet_sp` e `02_silver_dengue_clima_sp`. A exportação é opcional e aceita somente um caminho em `/Volumes`.

In [0]:
%pip install plotly==7.0.0 -q


In [ ]:
%run ./setup_pipeline

In [ ]:
for parametro in ("execucao_id", "versao_codigo"):
    dbutils.widgets.text(parametro, "")
EXECUCAO, REVISAO = iniciar_execucao(dbutils.widgets.get("execucao_id"), dbutils.widgets.get("versao_codigo"))


In [0]:
from pathlib import Path
from datetime import datetime, timezone
from base64 import b64decode
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown
from pyspark.sql import functions as F

ANOS = (2010, 2022)
SILVER = "dengue_sus.silver"
DESTINO = ""  # Ex.: /Volumes/<catalogo>/<schema>/<volume>/agregacao_sp

FONTES = {
    "mensal": f"{SILVER}.dengue_clima_sp_mes",
    "municipal": f"{SILVER}.dengue_sidra_municipio_ano_sp",
    "sinan": f"{SILVER}.sinan_sp_mes",
    "auditoria": f"{SILVER}.sinan_auditoria_sp",
    "desfechos_auditoria": f"{SILVER}.sinan_desfechos_auditoria_sp",
}
METRICAS = ("notificacoes", "casos_confirmados", "casos_graves", "obitos_dengue")
DESFECHOS = {
    "notificacoes": ("Notificações", "taxa_notificacoes_100k"),
    "casos_graves": ("Casos graves", "taxa_casos_graves_100k"),
    "obitos_dengue": ("Óbitos por dengue", "taxa_obitos_dengue_100k"),
}
MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
SANEAMENTO = {"agua_pct": "Água (%)", "esgoto_pct": "Esgoto (%)", "lixo_pct": "Lixo (%)"}
CLIMA = {
    "precip_media_estacoes_mm": ("Chuva mensal (mm)", "dias_precip"),
    "temp_media_c": ("Temperatura média (°C)", "dias_temp"),
    "umidade_media_pct": ("Umidade média (%)", "dias_umidade"),
}
CORES = {2010: "#2878b5", 2022: "#d77b29"}
FIGURAS = {}

def dados_portaveis(valor):
    if isinstance(valor, dict):
        if "bdata" in valor and "dtype" in valor:
            matriz = np.frombuffer(b64decode(valor["bdata"]), dtype=valor["dtype"])
            if "shape" in valor:
                matriz = matriz.reshape(tuple(int(n) for n in valor["shape"].split(",")))
            return matriz.tolist()
        return {chave: dados_portaveis(item) for chave, item in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [dados_portaveis(item) for item in valor]
    if hasattr(valor, "tolist"):
        return dados_portaveis(valor.tolist())
    return valor

def mostrar(nome, figura):
    figura.update_layout(template="plotly_white", font=dict(family="Arial, sans-serif", size=13), separators=",.")
    figura = go.Figure(dados_portaveis(figura.to_dict()))
    FIGURAS[nome] = figura
    displayHTML(figura.to_html(full_html=False, include_plotlyjs=True))

In [0]:
PRODUTORES = {
    "mensal": "02_silver_dengue_clima_sp.py", "municipal": "02_silver_dengue_sidra_sp.py",
    "sinan": "02_silver_sinan_sp.py", "auditoria": "02_silver_sinan_sp.py",
    "desfechos_auditoria": "02_silver_sinan_sp.py",
}
disponibilidade = pd.DataFrame([
    {"fonte": nome, "tabela": tabela, "notebook_produtor": PRODUTORES[nome],
     "disponivel": spark.catalog.tableExists(tabela)}
    for nome, tabela in FONTES.items()
])
display(disponibilidade)
ausentes = disponibilidade.loc[~disponibilidade["disponivel"]]
if not ausentes.empty:
    instrucoes = "; ".join(
        f"{linha.tabela}: execute notebooks/databricks/{linha.notebook_produtor} até o final"
        for linha in ausentes.itertuples()
    )
    raise ValueError(f"Tabelas Silver ausentes ou sem acesso. {instrucoes}")

mensal = ler_tabela(FONTES["mensal"]).filter(F.col("ano").isin(*ANOS)).limit(25).toPandas()
municipal = ler_tabela(FONTES["municipal"]).filter(F.col("ano").isin(*ANOS)).limit(1291).toPandas()
sinan = ler_tabela(FONTES["sinan"]).filter(F.col("ano").isin(*ANOS))
auditoria = (ler_tabela(FONTES["auditoria"]).filter(F.col("ano").isin(*ANOS))
             .groupBy("ano", "motivo").agg(F.sum("registros").alias("registros")).toPandas())
auditoria_desfechos = (ler_tabela(FONTES["desfechos_auditoria"])
                       .filter(F.col("ano").isin(*ANOS)).toPandas())

## 2. Critérios da integração

- **Notificação:** uma linha SINAN de residente em município válido de SP, sem filtro de classificação.
- **Confirmado:** critério laboratorial ou clínico-epidemiológico e classificação de dengue compatível com a ficha do ano.
- **Grave:** em 2010, dengue com complicações, febre hemorrágica ou síndrome do choque; em 2022, dengue com sinais de alarme ou dengue grave.
- **Óbito por dengue:** caso confirmado cuja evolução é óbito pelo agravo (`EVOLUCAO = 2`).
- **Taxas:** contagem dividida pela população do Censo do mesmo ano, vezes 100 mil.

As definições seguem a orientação do [Ministério da Saúde para tabulação de dengue](https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue). Os códigos brutos de classificação, critério e evolução permanecem na tabela de auditoria.

`*_desconhecidos` conta regras indeterminadas por campos nulos/vazios, sem somá-las aos eventos elegíveis. Códigos informados fora dos grupos elegíveis permanecem na auditoria. O contrato difere do histórico local, que não exige `CRITERIO`.

In [0]:
if len(mensal) != 24 or mensal.duplicated(["ano", "mes"]).any():
    raise ValueError("A Silver mensal deve conter 12 meses únicos em cada ano.")
if not mensal.groupby("ano")["mes"].apply(lambda meses: set(meses) == set(range(1, 13))).all():
    raise ValueError("A Silver mensal não contém todos os meses de 2010 e 2022.")
if len(municipal) != 1290 or municipal.duplicated(["ano", "codigo_municipio"]).any():
    raise ValueError("A Silver municipal deve conter 645 municípios únicos em cada ano.")
if not municipal.groupby("ano").size().eq(645).all():
    raise ValueError("Grade municipal incompleta por ano.")
if not municipal["codigo_municipio"].astype(str).str.fullmatch(r"35\d{4}").all():
    raise ValueError("Código municipal fora da grade de SP.")

for nome, quadro in (("mensal", mensal), ("municipal", municipal)):
    if quadro["populacao"].isna().any() or (quadro["populacao"] <= 0).any():
        raise ValueError(f"População ausente ou não positiva na base {nome}.")
    for metrica in (*METRICAS, *DESCONHECIDOS):
        taxa = f"taxa_{metrica}_100k"
        if quadro[metrica].isna().any() or (quadro[metrica] < 0).any():
            raise ValueError(f"{metrica}: contagem ausente ou negativa na base {nome}.")
        if metrica in METRICAS and not np.allclose(quadro[taxa], quadro[metrica] * 100000.0 / quadro["populacao"]):
            raise ValueError(f"{taxa}: cálculo inconsistente na base {nome}.")
    if ((quadro["casos_graves"] > quadro["casos_confirmados"])
        | (quadro["obitos_dengue"] > quadro["casos_confirmados"])
        | (quadro["casos_confirmados"] > quadro["notificacoes"])).any():
        raise ValueError(f"Hierarquia de desfechos inválida na base {nome}.")
for indicador in SANEAMENTO:
    if not municipal[indicador].between(0, 100.1).all():
        raise ValueError(f"Percentual municipal ausente ou inválido: {indicador}")

populacao_sp = municipal.groupby("ano")["populacao"].sum().astype("int64")
if not mensal.apply(lambda linha: linha["populacao"] == populacao_sp.loc[linha["ano"]], axis=1).all():
    raise ValueError("População mensal difere da soma dos 645 municípios.")
mensal = mensal.sort_values(["ano", "mes"]).reset_index(drop=True)
mensal["competencia"] = pd.to_datetime(dict(year=mensal["ano"], month=mensal["mes"], day=1))
mensal["clima_disponivel"] = mensal[list(CLIMA)].notna().all(axis=1)
for variavel, (_, dias) in CLIMA.items():
    dias_mes = mensal["competencia"].dt.days_in_month
    if not mensal["dias_mes"].eq(dias_mes).all() or not mensal[dias].between(0, dias_mes).all():
        raise ValueError(f"Cobertura diária inválida: {variavel}")
    minimo = dias_mes if variavel == "precip_media_estacoes_mm" else 0.8 * dias_mes
    if (mensal[variavel].notna() & (mensal[dias] < minimo)).any():
        raise ValueError(f"Clima publicado abaixo da cobertura mínima: {variavel}")
    mensal[f"cobertura_{variavel}_pct"] = 100 * mensal[dias] / dias_mes
meses_clima_comum = sorted(set.intersection(*[
    set(mensal.loc[(mensal["ano"] == ano) & mensal["clima_disponivel"], "mes"]) for ano in ANOS
]))
display(pd.DataFrame({
    "ano": ANOS, "meses": [12, 12], "municipios": [645, 645],
    "populacao": [populacao_sp.loc[ano] for ano in ANOS],
    "meses_clima_completo": [int(mensal.loc[mensal["ano"] == ano, "clima_disponivel"].sum()) for ano in ANOS],
}))
validar_leituras()
display(municipal.groupby("ano")[list(DESCONHECIDOS)].sum())


## 3. Auditoria e reconciliação

As contagens são reconciliadas entre as tabelas SINAN, climática e municipal. Datas inválidas ou fora do ano-fonte explicam a diferença entre o total municipal anual e os 12 meses do mesmo ano civil. A auditoria de desfechos permite inspecionar códigos não classificados e valores ausentes.

In [0]:
sinan_mensal = (sinan.filter(F.col("ano_calendario_notificacao") == F.col("ano"))
                 .select("ano", "mes", *CONTAGENS).toPandas().sort_values(["ano", "mes"]))
if len(sinan_mensal) != 24 or sinan_mensal.duplicated(["ano", "mes"]).any():
    raise ValueError("SINAN mensal não contém 12 meses únicos em cada ano.")
for metrica in CONTAGENS:
    if not np.array_equal(sinan_mensal[metrica].to_numpy(), mensal[metrica].to_numpy()):
        raise ValueError(f"{metrica}: painel climático difere da Silver SINAN mensal.")

totais_fonte = {m: {r["ano"]: r["total"] for r in sinan.groupBy("ano").agg(F.sum(m).alias("total")).collect()}
                for m in CONTAGENS}
totais_municipais = municipal.groupby("ano")[list(CONTAGENS)].sum()
for metrica in CONTAGENS:
    esperado = pd.Series(totais_fonte[metrica]).sort_index().astype("int64")
    observado = totais_municipais[metrica].sort_index().astype("int64")
    if not esperado.equals(observado):
        raise ValueError(f"{metrica}: total municipal não reconcilia com a Silver SINAN.")

totais_mensais = mensal.groupby("ano")[list(CONTAGENS)].sum()
fora_calendario = (auditoria.loc[auditoria["motivo"].isin(["data_invalida", "data_fora_do_ano"])]
                   .groupby("ano")["registros"].sum().reindex(ANOS, fill_value=0).astype("int64"))
if not (totais_municipais["notificacoes"] == totais_mensais["notificacoes"] + fora_calendario).all():
    raise ValueError("Notificações não reconciliam com datas inválidas ou fora do ano.")
display(auditoria.sort_values(["ano", "motivo"]))
display(pd.DataFrame({
    "ano": ANOS,
    "ano_fonte_municipios_validos": totais_municipais["notificacoes"].reindex(ANOS).to_numpy(),
    "mesmo_ano_civil": totais_mensais["notificacoes"].reindex(ANOS).to_numpy(),
    "datas_invalidas_ou_de_outros_anos": fora_calendario.to_numpy(),
}))
display(auditoria_desfechos.sort_values(
    ["ano", "classificacao_original", "criterio_original", "evolucao_original"], na_position="first"
))
print("Grade, denominadores, desfechos e reconciliação verificados.")

## 4. Bases mensal, anual e municipal

O resumo compara os volumes e taxas nos dois anos. `ano_fonte` inclui registros com datas inválidas ou de outros anos; `mesmo_ano` usa somente os 12 meses civis correspondentes. Gravidade e letalidade usam casos confirmados como denominador.

In [0]:
linhas = []
for ano in ANOS:
    m = mensal.loc[mensal["ano"] == ano]
    linha = {"UF": "SP", "ano": ano, "populacao": int(populacao_sp.loc[ano])}
    for metrica in METRICAS:
        linha[f"{metrica}_mesmo_ano"] = int(totais_mensais.loc[ano, metrica])
        linha[f"taxa_{metrica}_mesmo_ano_100k"] = (
            100000 * linha[f"{metrica}_mesmo_ano"] / linha["populacao"]
        )
        linha[f"{metrica}_ano_fonte"] = int(totais_municipais.loc[ano, metrica])
        linha[f"taxa_{metrica}_ano_fonte_100k"] = (
            100000 * linha[f"{metrica}_ano_fonte"] / linha["populacao"]
        )
    confirmados = linha["casos_confirmados_ano_fonte"]
    linha["casos_graves_por_100_confirmados"] = (
        100 * linha["casos_graves_ano_fonte"] / confirmados if confirmados else np.nan
    )
    linha["obitos_por_100_confirmados"] = (
        100 * linha["obitos_dengue_ano_fonte"] / confirmados if confirmados else np.nan
    )
    linha["chuva_anual_completa_mm"] = (
        m["precip_media_estacoes_mm"].sum()
        if m["dias_precip"].eq(m["dias_mes"]).all()
        and m["precip_media_estacoes_mm"].notna().all()
        else np.nan
    )
    for variavel in ("temp_media_c", "umidade_media_pct"):
        dias = CLIMA[variavel][1]
        linha[variavel] = (
            np.average(m[variavel], weights=m[dias])
            if m[variavel].notna().all()
            else np.nan
        )
    linhas.append(linha)
anual = pd.DataFrame(linhas)

indicadores_comparacao = [
    "taxa_notificacoes_ano_fonte_100k",
    "taxa_casos_confirmados_ano_fonte_100k",
    "taxa_casos_graves_ano_fonte_100k",
    "taxa_obitos_dengue_ano_fonte_100k",
    "casos_graves_por_100_confirmados",
    "obitos_por_100_confirmados",
]
comparacao = anual.set_index("ano")[indicadores_comparacao].T.rename(
    columns={2010: "2010", 2022: "2022"}
)
comparacao["diferenca_2022_menos_2010"] = comparacao["2022"] - comparacao["2010"]
comparacao["variacao_pct"] = np.where(
    comparacao["2010"] != 0,
    100 * comparacao["diferenca_2022_menos_2010"] / comparacao["2010"],
    np.nan,
)
display(mensal)
display(anual)
display(comparacao)
display(
    municipal.sort_values(["ano", "taxa_notificacoes_100k"], ascending=[True, False])
    .groupby("ano")
    .head(10)
)

## 5. Sazonalidade dos desfechos em SP

Cada painel usa sua própria escala de cor e compara os mesmos meses de 2010 e 2022.

In [0]:
fig = make_subplots(rows=3, cols=1, vertical_spacing=0.12,
                    subplot_titles=[f"{rotulo} por 100 mil" for rotulo, _ in DESFECHOS.values()])
for linha, (metrica, (_, taxa)) in enumerate(DESFECHOS.items(), start=1):
    matriz = mensal.pivot(index="ano", columns="mes", values=taxa).reindex(index=ANOS, columns=range(1, 13))
    fig.add_trace(go.Heatmap(
        x=MESES, y=[str(a) for a in ANOS], z=matriz.to_numpy(), colorscale="YlOrRd", showscale=False,
        hovertemplate="Ano: %{y}<br>Mês: %{x}<br>Taxa: %{z:.3f}/100 mil<extra></extra>",
    ), row=linha, col=1)
fig.update_layout(title="Dengue: taxas mensais em SP — 2010 × 2022", height=760)
mostrar("01_taxas_mensais", fig)

## 6. Painel temporal de dengue e clima

As linhas sobrepõem os dois anos para os três desfechos e para as variáveis meteorológicas, sem preencher lacunas de cobertura.

In [0]:
titulos = [f"{rótulo} por 100 mil" for rótulo, _ in DESFECHOS.values()] + [rótulo for rótulo, _ in CLIMA.values()]
fig = make_subplots(rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.035, subplot_titles=titulos)
series = [(taxa, "por 100 mil") for _, taxa in DESFECHOS.values()] + [
    ("precip_media_estacoes_mm", "mm"), ("temp_media_c", "°C"), ("umidade_media_pct", "%")
]
for linha, (coluna, unidade) in enumerate(series, start=1):
    for ano in ANOS:
        fatia = mensal.loc[mensal["ano"] == ano]
        fig.add_trace(go.Scatter(
            x=MESES, y=fatia[coluna].tolist(), mode="lines+markers", name=str(ano),
            legendgroup=str(ano), showlegend=linha == 1, connectgaps=False,
            line=dict(color=CORES[ano], dash="solid" if ano == 2010 else "dash"),
        ), row=linha, col=1)
    fig.update_yaxes(title_text=unidade, row=linha, col=1, rangemode="tozero" if linha <= 4 else "normal")
fig.update_layout(title="SP: dengue e clima ao longo de 2010 e 2022", height=1180,
                  legend=dict(orientation="h", y=1.04, x=1, xanchor="right"))
mostrar("02_painel_temporal", fig)

## 7. Dengue e saneamento nos municípios de SP

Os gráficos usam taxas anuais e mostram os anos em linhas separadas. As categorias de saneamento diferem entre os Censos; portanto, compare a distribuição municipal dentro de cada ano, sem interpretar a distância 2010–2022 como evolução do mesmo indicador.

In [0]:
for metrica, (rotulo, taxa) in DESFECHOS.items():
    fig = make_subplots(rows=2, cols=3, shared_yaxes="all", vertical_spacing=0.12,
                        subplot_titles=[f"{ano} — {nome}" for ano in ANOS for nome in SANEAMENTO.values()])
    for linha, ano in enumerate(ANOS, start=1):
        fatia = municipal.loc[municipal["ano"] == ano]
        for coluna, (indicador, nome) in enumerate(SANEAMENTO.items(), start=1):
            fig.add_trace(go.Scatter(
                x=fatia[indicador].tolist(), y=fatia[taxa].tolist(), mode="markers",
                text=fatia["municipio"].tolist(), customdata=fatia[["codigo_municipio", "populacao", metrica]].to_numpy(),
                marker=dict(size=6, opacity=0.45, color=CORES[ano]), showlegend=False,
                hovertemplate=("%{text}<br>Código: %{customdata[0]}<br>População: %{customdata[1]:,}"
                               f"<br>{nome}: %{{x:.1f}}%<br>{rotulo}: %{{customdata[2]:,}}"
                               "<br>Taxa: %{y:.3f}/100 mil<extra></extra>"),
            ), row=linha, col=coluna)
            fig.update_xaxes(title_text="% de domicílios", range=[0, 102], row=linha, col=coluna)
    fig.update_yaxes(title_text=f"{rotulo}/100 mil", autorange=True, rangemode="tozero", row=1, col=1)
    fig.update_yaxes(title_text=f"{rotulo}/100 mil", autorange=True, rangemode="tozero", row=2, col=1)
    fig.update_layout(title=f"{rotulo} e saneamento — municípios de SP", height=820)
    mostrar(f"03_{metrica}_saneamento_municipal", fig)

## 7.1. Dengue e clima nos meses de SP

A amostra é comum aos dois anos: somente meses em que chuva, temperatura e umidade atendem simultaneamente à cobertura mínima em 2010 e 2022.

In [0]:
amostra = mensal.loc[mensal["mes"].isin(meses_clima_comum)].copy()
display(Markdown(f"**Amostra comum: {len(meses_clima_comum)} meses por ano ({len(amostra)} observações).**"))
if meses_clima_comum:
    amostra["mes_nome"] = amostra["mes"].map(dict(enumerate(MESES, start=1)))
    for metrica, (rotulo, taxa) in DESFECHOS.items():
        fig = make_subplots(rows=2, cols=3, shared_yaxes="all", vertical_spacing=0.14,
                            subplot_titles=[f"{ano} — {nome}" for ano in ANOS for nome, _ in CLIMA.values()])
        for linha, ano in enumerate(ANOS, start=1):
            fatia = amostra.loc[amostra["ano"] == ano]
            for coluna, (variavel, (nome, _)) in enumerate(CLIMA.items(), start=1):
                fig.add_trace(go.Scatter(
                    x=fatia[variavel].tolist(), y=fatia[taxa].tolist(), text=fatia["mes_nome"].tolist(),
                    mode="markers", marker=dict(size=10, color=CORES[ano]), showlegend=False,
                    hovertemplate=f"%{{text}}<br>{nome}: %{{x:.1f}}<br>{rotulo}: %{{y:.3f}}/100 mil<extra></extra>",
                ), row=linha, col=coluna)
                fig.update_xaxes(title_text=nome, row=linha, col=coluna)
        fig.update_yaxes(title_text=f"{rotulo}/100 mil", row=1, col=1, rangemode="tozero")
        fig.update_yaxes(title_text=f"{rotulo}/100 mil", row=2, col=1, rangemode="tozero")
        fig.update_layout(title=f"{rotulo} e clima — SP", height=800)
        mostrar(f"05_{metrica}_clima_mensal", fig)
else:
    display(Markdown("Não há meses comuns com as três variáveis climáticas elegíveis."))

## 8. Cobertura meteorológica

O mapa evidencia a cobertura diária usada em cada variável e ano. Temperatura e umidade exigem 80% dos dias válidos; chuva mensal exige todos os dias do mês.

In [0]:
display(mensal[["ano", "mes", "dias_mes", "dias_precip", "dias_temp", "dias_umidade",
                "estacoes_medias_presentes", "clima_disponivel"]])
linhas_cobertura, matriz_cobertura = [], []
for ano in ANOS:
    fatia = mensal.loc[mensal["ano"] == ano].sort_values("mes")
    for variavel, (rotulo, _) in CLIMA.items():
        linhas_cobertura.append(f"{ano} — {rotulo}")
        matriz_cobertura.append(fatia[f"cobertura_{variavel}_pct"].tolist())
fig = go.Figure(go.Heatmap(
    x=MESES, y=linhas_cobertura, z=matriz_cobertura, zmin=0, zmax=100,
    colorscale=[[0, "#ededed"], [0.5, "#8bc4ac"], [1, "#146c50"]], colorbar_title="Dias válidos (%)",
    hovertemplate="%{y}<br>Mês: %{x}<br>Dias válidos: %{z:.1f}%<extra></extra>",
))
fig.update_layout(title="Cobertura diária das estações disponíveis — SP", height=520,
                  margin=dict(l=260, r=40, t=80, b=60))
mostrar("04_cobertura_inmet", fig)

## 9. Exportação opcional para um Volume

Quando `DESTINO` é configurado, os dados validados, a auditoria e todos os gráficos são gravados em uma pasta nova `comparacao_2010_2022_<identificador>`. A exportação só está completa quando contém `_SUCCESS`.

In [0]:
validar_leituras()
metadados = {
    "execucao_id": EXECUCAO, "versao_codigo_declarada": REVISAO,
    "versoes_delta": VERSOES, "entradas": FONTES_EXECUCAO,
    "anos": ANOS, "UF": "SP", "fontes": FONTES,
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "notificacoes": "residentes, independentemente da classificação final",
    "confirmados": "classificação de dengue e critério laboratorial ou clínico-epidemiológico",
    "graves": "2010: classes 2, 3 e 4; 2022: classes 11 e 12; apenas confirmados",
    "obitos": "casos confirmados com EVOLUCAO=2; não são pessoas deduplicadas",
    "periodo_municipal": "ano-fonte SINAN, incluindo datas inválidas e de outros anos",
    "periodo_mensal": "ano civil da notificação igual ao ano-fonte SINAN",
    "desconhecidos": "flags indeterminadas por nulos/vazios; contagens separadas dos elegíveis",
    "saneamento": "definições Silver próprias de cada Censo; sem harmonização temporal",
    "clima": "estações disponíveis, com composição variável; dias meteorológicos UTC",
    "cobertura_minima_mensal": {"chuva": 1.0, "temperatura_umidade": 0.8},
}
display(pd.DataFrame.from_dict(FONTES_EXECUCAO, orient="index").reset_index(names="origem"))
if DESTINO:
    raiz = Path(DESTINO)
    if not raiz.is_absolute() or len(raiz.parts) < 5 or raiz.parts[1] != "Volumes" or ".." in raiz.parts:
        raise ValueError("DESTINO deve estar em /Volumes/<catalogo>/<schema>/<volume>/...")
    destino = raiz / f"comparacao_2010_2022_{uuid4().hex}"
    # Uma pasta nova por exportação evita misturar resultados com uma anterior.
    destino.mkdir(parents=True, exist_ok=True)
    quadros = {
        "mensal_sp": mensal, "anual_sp": anual, "comparacao_sp": comparacao.reset_index(names="indicador"),
        "municipal_sp": municipal, "auditoria_sinan_resumo": auditoria,
        "auditoria_desfechos": auditoria_desfechos,
    }
    for nome, quadro in quadros.items():
        quadro.to_csv(destino / f"{nome}.csv", index=False)
    pasta_figuras = destino / "figuras"
    pasta_figuras.mkdir(exist_ok=True)
    for nome, figura in FIGURAS.items():
        figura.write_html(pasta_figuras / f"{nome}.html", include_plotlyjs=True)
    validar_leituras()
    (destino / "metadados.json").write_text(json.dumps(metadados, ensure_ascii=False, indent=2), encoding="utf-8")
    (destino / "_SUCCESS").write_text(EXECUCAO, encoding="utf-8")
    display(pd.DataFrame({"arquivo": [str(p) for p in sorted(destino.rglob("*")) if p.is_file()]}))
else:
    print("Exportação desativada. Configure DESTINO com um diretório em um Volume para salvar os resultados.")

## Limites de interpretação e validação

- As associações dos gráficos são descritivas e não demonstram causalidade.
- Os dois anos usam classificações SINAN diferentes, harmonizadas apenas nos grupos documentados de confirmação e gravidade.
- Óbitos são registros confirmados com evolução informada como óbito pelo agravo; valores ausentes permanecem na auditoria.
- Os indicadores de saneamento mantêm as definições de cada Censo e não formam uma série temporal harmonizada.
- O clima representa o conjunto de estações disponível na UF, não a exposição individual ou municipal.
- Taxas de eventos raros, sobretudo óbitos em municípios pequenos, são instáveis e devem ser lidas com as contagens.
- A execução final deve ser validada no Databricks, onde estão as tabelas Spark e o `displayHTML`.